# Lab 2.2. From the web table to the image: the reservoir on two dates

**Module 2, Session 2. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

**The question of the session: how much water is in the reservoir?** Part 1 reads the answer a
basin authority publishes in a web bulletin: the claim. Part 2 measures the same surface from a
satellite image: the check. Exercise 8 compares them.

By the end of this notebook you should be able to:

1. Extract a table from a web page with a two-level header and Spanish number conventions.
2. Validate the table against a control figure the page itself publishes.
3. Convert Sentinel-2 digital numbers into reflectance by reading the metadata, and quantify the error of not doing so.
4. Compute a normalised spectral index and justify the threshold.
5. Estimate the water surface on two dates and compare the result with an independent source.

**Before handing in:** Kernel, Restart & Run All.

## 0. Setup

This cell locates the course folder wherever the notebook is running: on your own machine, in
Colab from the course repository, or in Colab from the shared Drive folder. Run it first and
check that the file listing appears.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")
    root = Path("/content/acs-mod2")               # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root
    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

### Installs (Colab only)

`rasterio` and `scikit-image` are not preinstalled in Colab; the cell installs them if needed.
It takes about a minute. Run it at the start of the break.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg, mod in [("rasterio", None), ("scikit-image", "skimage"), ("lxml", None),
                 ("html5lib", None), ("geopandas", None), ("pyarrow", None)]:
    ensure(pkg, mod)
print("ready")

In [ ]:
import json
import pandas as pd, numpy as np, rasterio, matplotlib
import matplotlib.pyplot as plt
for m in (pd, np, rasterio, matplotlib):
    print(f"{m.__name__:<10}", m.__version__)

## Part 1. The claim: a table on a web page

An HTML page is a tree of tags. A table is a branch: `table > thead/tbody > tr > th/td`.
`pandas.read_html` walks the tree and returns **a list** with every table it finds.

Today we work on a saved copy. Fetching live pages, HTTP headers, `robots.txt` and terms of use
are next week's session.

### Exercise 1. Locate the table

Read the bulletin and find how many tables it contains. Then keep the one you need with the
`match` argument, which filters by a text contained in the table.

In [ ]:
html_path = DATA / "boletin_embalses.html"
all_tables = pd.read_html(html_path)
print("Tables found:", len(all_tables))
for i, t in enumerate(all_tables):
    print(f"  [{i}] {t.shape}  columns: {list(t.columns)[:3]}")

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(raw.shape)
raw.head(3)

In [ ]:
assert raw.shape[0] == 10, f"Expected 10 rows (9 reservoirs and the total), got {raw.shape[0]}"
assert isinstance(raw.columns, pd.MultiIndex), "With header=[0,1] the columns should be a MultiIndex"
print("Checks passed.")

### Exercise 2. Clean

The `rowspan` and `colspan` tags of the header produce a two-level column index. Flatten it, take
the totals row out, strip the footnote marker `[1]` from the reservoir name, and leave the result in
`reservoirs`, with columns `reservoir`, `capacity_hm3`, `fill_pct` and `surface_ha`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
reservoirs

In [ ]:
assert list(reservoirs.columns) == ["reservoir", "capacity_hm3", "fill_pct", "surface_ha"]
assert len(reservoirs) == 9, f"Expected 9 reservoirs, got {len(reservoirs)}"
assert not reservoirs["reservoir"].str.contains(r"\[").any(), "A footnote marker remains"
assert reservoirs["capacity_hm3"].dtype.kind == "f", "Capacity should be numeric"
print("Checks passed.")

### Exercise 3. Validate against the control figure

Almost every published table includes its own total. It is the cheapest check there is and almost
nobody does it. Recompute the sum of capacities and compare with the total the page prints.

In [ ]:
published_total = float(total_row["capacity_hm3"].iloc[0])
# YOUR CODE HERE
raise NotImplementedError
print(f"Published total: {published_total:8.1f} hm3")
print(f"Computed total : {computed_total:8.1f} hm3")
print(f"Relative error : {rel_error:.2%}")

In [ ]:
assert rel_error < 0.005, f"Discrepancy with the published total: {rel_error:.2%}"
print("Checks passed: the extraction reproduces the control figure.")

In [ ]:
bulletin_surface = float(
    reservoirs.loc[reservoirs["reservoir"] == "Presa de la Hoz", "surface_ha"].iloc[0])
print(f"The bulletin declares {bulletin_surface:.1f} ha of water surface at Presa de la Hoz.")
print("We will check it against the satellite image at the end of the notebook.")

## Part 2. The check: from pixels to hectares

A multispectral scene is a stack of surfaces: the field model of session 1, repeated once per band.
Everything you know about a raster still holds; what is new is that the values are not colours or
counts, but reflectance encoded as integers. Four steps turn the scene into a surface, exactly as
on the slide:

1. **Read reflectance** from the digital numbers, with the offset in the metadata (exercise 4).
2. **Compute NDWI**, water positive and land negative (exercise 6).
3. **Threshold and mask** clouds and shadows (exercise 7).
4. **Count and convert**: water pixels times 400 m2, divided by 10 000 (exercise 7).

Exercise 5, the colour composites, is a guided look at the scene before the arithmetic.

### Exercise 4. Step 1: from digital number to reflectance

Sentinel-2 products store reflectance as integers. Since processing baseline 04.00 (25 January
2022) a **radiometric offset** is applied per band, so that negative reflectances over very dark
surfaces can be encoded without clipping. The correct conversion is

$$\rho = \frac{DN + \text{BOA\_ADD\_OFFSET}}{\text{QUANTIFICATION\_VALUE}}$$

and both values are read from the product metadata, never hard-coded.

Write `read_band`, which returns the reflectance of one band on one date.

In [ ]:
from IPython.display import Image as IPImage, display
fig_path = BASE / "figuras" / "pipeline_pixels_to_hectares.png"
if fig_path.exists():
    display(IPImage(filename=str(fig_path), width=900))
else:
    print("The pipeline figure is in the slides; the notebook reproduces it step by step.")

In [ ]:
DATES = {"drought": "20230812", "full": "20250427"}

def metadata(date):
    with open(DATA / f"s2_{date}_MTD.json", encoding="utf-8") as f:
        return json.load(f)

print(json.dumps(metadata(DATES["full"]), indent=2, ensure_ascii=False)[:520])

In [ ]:
def read_band(date, band):
    '''Surface reflectance of one band, according to the product metadata.'''
    # YOUR CODE HERE
    raise NotImplementedError

ref = {b: read_band(DATES["full"], b) for b in ["B02", "B03", "B04", "B08"]}
for b, a in ref.items():
    print(f"{b}: min {a.min():6.3f}  mean {a.mean():6.3f}  max {a.max():6.3f}")

In [ ]:
assert ref["B08"].min() < 0.05, "Near-infrared over water must be very low"
assert ref["B08"].max() < 1.5, "Reflectances above 1.5 mean the conversion is missing"
assert abs(ref["B04"].mean() - 0.13) < 0.08, "Mean values outside the plausible range"
print("Checks passed.")

**Predict before you run:** if you divide by 10000 and ignore the offset, will water look
darker or brighter in the near infrared than it really is?

Quantify the error, using the scene classification band (SCL), which labels every pixel, to keep
only water pixels (class 6).

In [ ]:
with rasterio.open(DATA / f"s2_{DATES['full']}_SCL.tif") as src:
    scl = src.read(1)
water_scl = scl == 6

with rasterio.open(DATA / f"s2_{DATES['full']}_B08.tif") as src:
    dn_b08 = src.read(1).astype("float32")
naive = dn_b08 / 10000.0

print(f"Mean NIR over water, correct conversion : {ref['B08'][water_scl].mean():.4f}")
print(f"Mean NIR over water, dividing by 10000  : {naive[water_scl].mean():.4f}")
print(f"\nWater appears {naive[water_scl].mean() / ref['B08'][water_scl].mean():.0f} times "
      f"brighter in the near infrared than it is.")

### Exercise 5. Colour composites *(guided)*

Given, to save time: a true-colour composite (red, green, blue) and a false-colour infrared one
(near infrared, red, green), stretched between the 2nd and 98th percentiles so that the brightest
pixel does not flatten the scale. Run it and answer: what is red in the right-hand image?

In [ ]:
def stretch(a, p=(2, 98)):
    lo, hi = np.nanpercentile(a, p)
    return np.clip((a - lo) / (hi - lo), 0, 1)

true_colour = np.dstack([stretch(ref["B04"]), stretch(ref["B03"]), stretch(ref["B02"])])
false_colour = np.dstack([stretch(ref["B08"]), stretch(ref["B04"]), stretch(ref["B03"])])

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
axes[0].imshow(true_colour); axes[0].set_title("True colour (B04, B03, B02)")
axes[1].imshow(false_colour); axes[1].set_title("False-colour infrared (B08, B04, B03)")
for a in axes: a.set_axis_off()
plt.tight_layout()
print("In false colour healthy vegetation is red and water almost black:")
print("near infrared is very high over vegetation and almost nil over water.")

### Exercise 6. Step 2: normalised indices

A normalised difference index has the form $(A-B)/(A+B)$, lies between −1 and 1 and is little
affected by the overall brightness of the scene.

- **NDVI** = (NIR − Red) / (NIR + Red), vegetation. Rouse et al. (1974).
- **NDWI** = (Green − NIR) / (Green + NIR), open water. McFeeters (1996).

There is another index also called NDWI, Gao (1996), which uses near and short-wave infrared and
measures water in leaves. They are not interchangeable: when you cite one, say which.

Implement them, taking care not to divide by zero.

In [ ]:
def normalised_index(a, b):
    # YOUR CODE HERE
    raise NotImplementedError

ndvi = normalised_index(ref["B08"], ref["B04"])
ndwi = normalised_index(ref["B03"], ref["B08"])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, arr, title, cmap in zip(axes, [ndvi, ndwi], ["NDVI", "NDWI (McFeeters)"], ["RdYlGn", "BrBG"]):
    im = ax.imshow(arr, cmap=cmap, vmin=-1, vmax=1); ax.set_title(title); ax.set_axis_off()
    plt.colorbar(im, ax=ax, shrink=.75)
plt.tight_layout()

In [ ]:
assert -1.01 <= ndvi.min() and ndvi.max() <= 1.01
assert ndwi[water_scl].mean() > 0.3, "NDWI over water should be clearly positive"
assert ndvi[scl == 4].mean() > 0.25, "NDVI over vegetation should be positive"
print("Checks passed.")
print(f"Mean NDWI over water     : {ndwi[water_scl].mean():.3f}")
print(f"Mean NDVI over vegetation: {ndvi[scl == 4].mean():.3f}")

### Exercise 7. Steps 3 and 4: threshold, mask and surface on two dates

Separating water from non-water needs a threshold. Three ways to choose it, in increasing order of
defensibility: zero, because the index is symmetric; a value from the literature, such as 0.3 to
avoid false positives in urban areas; or a value derived from the histogram of this very image with
Otsu's method (1979), which finds the threshold that minimises the variance within each class.

Write `water_surface`, which for a given date:

1. reads the green and near-infrared bands with the correct conversion,
2. computes the NDWI,
3. discards cloud and cloud-shadow pixels using the SCL band (classes 3, 8 and 9),
4. thresholds with Otsu,
5. returns the surface in hectares, knowing the pixel is 20 × 20 m.

In [ ]:
from skimage.filters import threshold_otsu
PIXEL_M = 20.0

def water_surface(date):
    '''Water surface in hectares, the mask used and the Otsu threshold.'''
    # YOUR CODE HERE
    raise NotImplementedError

results = {k: water_surface(d) for k, d in DATES.items()}
for k, (ha, _, thr) in results.items():
    print(f"{k:>8}: {ha:7.1f} ha   (Otsu threshold {thr:+.3f})")
reduction = 1 - results["drought"][0] / results["full"][0]
print(f"\nLoss of water surface in the drought episode: {reduction:.1%}")

In [ ]:
ha_full = results["full"][0]
ha_drought = results["drought"][0]
assert 250 < ha_full < 360, f"Surface on the full date outside expectations: {ha_full:.1f}"
assert 100 < ha_drought < 180, f"Surface in drought outside expectations: {ha_drought:.1f}"
assert ha_full > ha_drought * 1.8, "The difference between dates should be very marked"
assert -0.6 < results["full"][2] < 0.4, "Unexpected Otsu threshold"
print("Checks passed.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
axes[0].hist(ndwi.ravel(), bins=120, color="#546E7A")
for u, c, lab in [(0.0, "#90A4AE", "zero"), (0.3, "#B0BEC5", "literature 0.3"),
                  (results["full"][2], "#964BFF", f"Otsu {results['full'][2]:+.3f}")]:
    axes[0].axvline(u, color=c, linewidth=2, label=lab)
axes[0].set_title("NDWI histogram, 2025"); axes[0].legend(fontsize=8); axes[0].grid(alpha=.3)
for ax, (k, (ha, m, _)) in zip(axes[1:], results.items()):
    ax.imshow(m, cmap="Blues"); ax.set_axis_off()
    ax.set_title(f"{k}  ({DATES[k][:4]}-{DATES[k][4:6]})  {ha:.0f} ha")
plt.tight_layout()
print("The histogram is bimodal: the large left peak is land, the small right one is water.")
print("Otsu finds the valley between them. When a histogram is not bimodal, Otsu still returns")
print("a number, and that number means nothing: look at the histogram before applying it.")

### Exercise 8. The claim against the check

The bulletin published a water surface for Presa de la Hoz, obtained by remote sensing on the same
date. Compare your result with that figure.

This is the first data-fusion exercise of the module: two independent routes, a web table and an
image, that must arrive at the same number.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(f"Bulletin : {bulletin_surface:7.1f} ha")
print(f"Image    : {ha_full:7.1f} ha")
print(f"Relative difference: {discrepancy:.2%}")

In [ ]:
assert discrepancy < 0.10, f"Discrepancy with the independent source too large: {discrepancy:.1%}"
print("Checks passed: the two sources agree within 10 %.")

_Answer:_ if the discrepancy had been 40 %, which three hypotheses would you check first, and in
which order?

### Extension A. The two mistakes at once

Repeat the surface calculation with both wrong decisions together: divide by 10000 without the
offset, and use the fixed literature threshold of 0.3.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(f"Correct conversion, threshold 0.3 : {ha_good:6.1f} ha")
print(f"Divide by 10000, threshold 0.3    : {ha_bad:6.1f} ha")

In [ ]:
assert ha_bad < 5, "With the offset ignored and a fixed threshold almost no water should be detected"
print("\nThe reservoir disappears. A metadata error, no exception, and a completely wrong")
print("conclusion about the state of the basin.")
print("\nAn honest nuance: with a threshold derived from the histogram (Otsu) the sign of the")
print("index is preserved and the result would have been almost right. The offset is critical")
print("for absolute values and fixed thresholds, not for ranking.")

### Extension B. Clip to the reservoir polygon

`embalse.gpkg` holds the polygon of the reservoir basin. Clip the mask with it to exclude ponds
outside the reservoir, and recompute the surface.

In [ ]:
import geopandas as gpd
from rasterio.mask import mask as clip_mask
# YOUR CODE HERE
raise NotImplementedError
print(f"Surface inside the basin: {ha_basin:.1f} ha")
print(f"Outside the basin       : {ha_full - ha_basin:.1f} ha")

---

## Take-aways

- `read_html` returns a list, not a table; `match` and `header` do the fine work.
- Almost every published table carries its own control figure. Recomputing it costs one line.
- Metadata say how the numbers are to be read. In Sentinel-2 that includes a radiometric offset
  which, ignored, can make a reservoir disappear.
- A normalised index needs a threshold, and the threshold must be defensible: Otsu derives it
  from the image itself.
- When two independent sources reach the same number, confidence in both goes up. When they do
  not, the interesting work begins.

**References.** Rouse et al. (1974); McFeeters (1996); Gao (1996); Otsu (1979).
ESA, Sentinel-2 MSI Level-2A technical guide.